# Production plan with retailers

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ula-uab/lscm-decision-making/blob/main/cases/production_plan/notebooks/retailers.ipynb)

A company assembles electric bicycles in three plants and plans how many bikes each plant assembles each month from January to June. In the fourth version of the model, M3, the plants and the central warehouse are part of a distribution network in Spain, and the local warehouses receive the orders. With forecast 5, M3 has no solution: the orders cannot all be delivered by the end of June. This notebook uses the fifth version, M4. The orders come from six retailers, a new stage at the end of the network, and part of the demand that a retailer does not receive in its month is lost: the retailer buys those bikes from a competitor.

Contents of the notebook:

1. Retailers, their arcs and forecast 5.
2. Linear model with retailers.
3. Optimal plan under forecast 5.
4. Four tests, each with one datum changed: the backorder share of a retailer, the lost-sale cost, the orders of a retailer in one month and the transport cost of an arc to a retailer.

The cells run in order, with ▶ or *Shift + Enter*. Some cells have a form (sliders and drop-down lists): after changing a value, run the cell again. Outside Colab the form is a line such as `month = "May"`: edit the value and run the cell.

The table gives the symbols of M0 to M4, with the term that names each concept. The column «From» gives the version where each symbol first appears.

| Symbol | Meaning | Term | Unit | From |
|---|---|---|---|---|
| $F$, $f$ | Plants | — | — | M0 |
| $T$, $t$ | Months, January to June; $t = 0$ is the end of December | — | — | M0 |
| $\tau$ | Last month of the horizon | — | — | M1 |
| $V$, $i$ | Nodes of the network: plants, warehouses, the rail terminal and the retailers | — | — | M3 |
| $A$, $a$ | Arcs of the network; the arc $a = \langle i, j \rangle$ goes from its origin node $i$ to its destination node $j$ | — | — | M3 |
| $A^{\text{in}}_j$; $A^{\text{out}}_i$ | Arcs that enter node $j$; arcs that leave node $i$ | — | — | M3 |
| $R$, $r$ | Customers of the manufacturer that sell the bikes to the final customers; the demand nodes of M4 | retailer | — | M4 |
| $c_f$ | Unit assembly cost at plant $f$ | — | €/bike | M0 |
| $K_{ft}$ | Assembly capacity of plant $f$ in month $t$ | — | bikes/month | M0 |
| $d_{rt}$ | Orders of retailer $r$ in month $t$ | — | bikes/month | M3 |
| $c^h_i$ | Cost of holding one bike in stock for one month at node $i$ | holding cost | €/(bike·month) | M1 |
| $h_{i0}$ | Stock at node $i$ at the end of December | — | bikes | M1 |
| $h^{\text{end}}_i$ | Stock that node $i$ must hold at the end of the horizon | required ending stock | bikes | M1 |
| $K^h_i$ | Number of bikes that node $i$ can hold | storage capacity | bikes | M2 |
| $c^b_r$ | Cost of one bike backordered for one month at retailer $r$ | backorder cost | €/(bike·month) | M2 |
| $c^x_a$ | Cost of carrying one bike along arc $a$ | transport cost | €/bike | M3 |
| $K^x_a$ | Maximum flow on arc $a$ in a month | arc capacity | bikes/month | M3 |
| $K^n_i$ | Maximum flow that can enter node $i$ in a month | node capacity | bikes/month | M3 |
| $\beta_r$ | Share of the demand that retailer $r$ does not receive in its month that is backordered; the rest, $1 - \beta_r$, is lost sales | backorder share | — (between 0 and 1) | M4 |
| $c^l_r$ | Cost of one lost sale at retailer $r$: the price that the retailer pays for a bike | lost-sale cost | €/bike | M4 |
| $p_{ft}$ | Bikes assembled at plant $f$ in month $t$ | — | bikes/month | M0 |
| $h_{it}$ | Bikes in stock at node $i$ at the end of month $t$ | — | bikes | M1 |
| $x_{at}$ | Bikes shipped on arc $a$ in month $t$ | shipments | bikes/month | M1 |
| $s_{rt}$ | Bikes that retailer $r$ receives in month $t$, for orders of that month or for backorders | deliveries | bikes/month | M2 |
| $b_{rt}$ | Orders of retailer $r$ not yet delivered at the end of month $t$ | backorders | bikes | M2 |
| $l_{rt}$ | Orders of retailer $r$ in month $t$ that it buys from a competitor | lost sales | bikes/month | M4 |

The [document of this version](https://github.com/ula-uab/lscm-decision-making/blob/main/cases/production_plan/retailers.md) explains all the data and results. The places and distances are real; the other data are invented. Running the notebook is optional: it is support material, not part of the assessment.

In [ ]:
# @title Setup { display-mode: "form" }
# In Google Colab, install the example from GitHub (it takes a few seconds).
# On your own computer, install the repository environment instead (see README).
import sys
if "google.colab" in sys.modules:
    url = "git+https://github.com/ula-uab/lscm-decision-making#subdirectory=cases/production_plan"
    %pip install -q {url}
    # A session that already had the example keeps its old code unless it is reinstalled
    %pip install -q --force-reinstall --no-deps {url}
    # and unless its old code is removed from memory, so that the import below reads
    # the new one: no need to restart the session
    import importlib
    for name in [m for m in sys.modules if m == "plants" or m.startswith("plants.")]:
        del sys.modules[name]
    importlib.invalidate_caches()
from plants import retailers as rt

## 1. Retailers, their arcs and forecast 5

The first table gives the place of each retailer. The second gives the arcs from the local warehouses to the retailers: each retailer can receive from two local warehouses, at different transport costs. The third gives, for each retailer, its share of forecast 5, its backorder share, its lost-sale cost and its orders of each month. The rest of the network keeps the data of M3.

In [ ]:
# @title Retailers, their arcs and forecast 5 { display-mode: "form" }
rt.show_data()

## 2. Linear model with retailers

$$
\begin{aligned}
\min \quad & \sum_{t \in T} \Big( \sum_{f \in F} c_f\, p_{ft} + \sum_{i \in V} c^h_i\, h_{it} + \sum_{a \in A} c^x_a\, x_{at} + \sum_{r \in R} \left( c^b_r\, b_{rt} + c^l_r\, l_{rt} \right) \Big) \\
\text{s.t.} \quad & h_{i,t-1} + p_{it} + \sum_{a \in A^{\text{in}}_i} x_{at} = \sum_{a \in A^{\text{out}}_i} x_{at} + s_{it} + h_{it} && \forall i \in V,\ t \in T && \text{(stock balance at each node)} \\
& \sum_{a \in A^{\text{out}}_i} x_{at} + s_{it} \le h_{i,t-1} + p_{it} + \sum_{a \in A^{\text{in}}_i} x_{at} && \forall i \in V,\ t \in T && \text{(shipment limit at each node)} \\
& b_{r,t-1} + d_{rt} = s_{rt} + b_{rt} + l_{rt} && \forall r \in R,\ t \in T && \text{(backorder balance)} \\
& b_{rt} = \beta_r \left( b_{r,t-1} + d_{rt} - s_{rt} \right) && \forall r \in R,\ t < \tau && \text{(backorder share)} \\
& x_{at} \le K^x_a && \forall a \in A,\ t \in T && \text{(arc capacity)} \\
& \sum_{a \in A^{\text{in}}_i} x_{at} \le K^n_i && \forall i \text{ with a node capacity},\ t \in T && \text{(node capacity)} \\
& h_{it} \le K^h_i && \forall i \in V,\ t \in T && \text{(storage capacity)} \\
& p_{ft} \le K_{ft} && \forall f \in F,\ t \in T && \text{(assembly capacity)} \\
& b_{r\tau} = 0 \quad \forall r \in R, \qquad h_{i\tau} \ge h^{\text{end}}_i \quad \forall i \in V && && \text{(backorders and required ending stock at the end)} \\
& p_{ft},\ h_{it},\ x_{at},\ s_{rt},\ b_{rt},\ l_{rt} \ge 0
\end{aligned}
$$

with $h_{i0}$ given and $b_{r0} = 0$. Only the plants assemble ($p_{it} = 0$ at any other node), and only the retailers receive deliveries ($s_{it} = 0$ at any other node); the retailers hold no stock. In the backorder balance, the backorders at the start of the month plus the orders of the month equal the deliveries, plus the backorders at the end of the month, plus the lost sales. From January to May, of the demand that a retailer does not receive in the month, the share $\beta_r$ is backordered and the rest is lost sales. At the end of June no order is backordered, and the orders not delivered by then are lost sales.

## 3. Optimal plan under forecast 5

The cell first gives the reason why M3 has no solution under forecast 5. Then it solves M4 and gives the bikes assembled at each plant, the flow on each arc, the stock of each node, and the orders, deliveries, backorders and lost sales of each retailer, with the cost of the plan and the total lost sales. Below, the **shadow price** of each capacity that is full in some month (€/bike). The figure on the left draws the network with the flow on each arc in the month you choose; an arc whose capacity is full is drawn in red. The one on the right gives the lost sales of each retailer in each month.

Choose May. Which retailers have lost sales, and why those? Which retailer receives from two local warehouses, and why? Which plants have free capacity, and in which months?

In [ ]:
# @title Optimal plan under forecast 5 { display-mode: "form" }
month = "May"  # @param ["Jan", "Feb", "Mar", "Apr", "May", "Jun"]
rt.retailer_plan(month);

## 4. Backorder share of a retailer

Choose a retailer and its backorder share. Of the demand that the retailer does not receive in a month, that share is backordered and the rest is lost sales. The cell gives the lost sales of each retailer, the bikes assembled at each plant and the figure for May.

Lower the backorder share of Granada to 0, then raise it to 1. Which retailers lose sales in each case? Does the total of lost sales change, and do the plants assemble more?

In [ ]:
# @title Optimal plan for a backorder share of a retailer { display-mode: "form" }
retailer = "Retailer 6"  # @param ["Retailer 1", "Retailer 2", "Retailer 3", "Retailer 4", "Retailer 5", "Retailer 6"]
backorder_share = 0.4  # @param {type:"slider", min:0, max:1, step:0.1}
rt.backorder_share_test(retailer, backorder_share);

## 5. Lost-sale cost

The slider sets the lost-sale cost, the price that every retailer pays for a bike. Each lost sale adds that price to the cost of the plan, and each bike delivered adds what assembling, holding and transporting it costs. The cell shows the same tables as the previous one.

Lower the lost-sale cost from 700. Below what price do the lost sales grow, and at which retailers first? What happens to the assembly of plant 3?

In [ ]:
# @title Optimal plan for a lost-sale cost { display-mode: "form" }
lost_sale_cost = 700  # @param {type:"slider", min:300, max:900, step:20}
rt.lost_sale_cost_test(lost_sale_cost);

## 6. Orders of a retailer in one month

Choose a retailer, a month and the orders it adds to forecast 5 in that month, for example because of a promotion. The cell shows the same tables, with the figure for that month.

Add 100 orders to Sabadell in January, then in May. Which plant assembles the extra bikes in January? Where do they come from in May, when the plants are at their capacity? Does it change anything to add them to Córdoba instead?

In [ ]:
# @title Optimal plan for the orders of a retailer in one month { display-mode: "form" }
retailer = "Retailer 2"  # @param ["Retailer 1", "Retailer 2", "Retailer 3", "Retailer 4", "Retailer 5", "Retailer 6"]
month = "Jan"  # @param ["Jan", "Feb", "Mar", "Apr", "May", "Jun"]
extra_orders = 100  # @param {type:"slider", min:0, max:200, step:10}
rt.promotion_test(retailer, month, extra_orders);

## 7. Transport cost of an arc to a retailer

Choose one of the arcs from a local warehouse to a retailer, arcs 14 to 25 of the list, and its transport cost. The cell first gives the flow on the twelve arcs to the retailers, then the same tables as the previous ones.

Make arc 17, from Tarragona to Sabadell, cheaper than arc 16, from Girona to Sabadell. Which local warehouse supplies Sabadell then? Why does Sabadell still receive some bikes from Girona in April and May?

In [ ]:
# @title Optimal plan for a transport cost of an arc to a retailer { display-mode: "form" }
arc = 17  # @param [14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25]
transport_cost = 1.05  # @param {type:"slider", min:0, max:7, step:0.05}
rt.transport_cost_test(arc, transport_cost);

The [document of this version](https://github.com/ula-uab/lscm-decision-making/blob/main/cases/production_plan/retailers.md) explains all the numbers and has a script that prints them.